# 02 — TTT Evaluation on ARC-AGI-2 (Colab Pro)

Loads a slim checkpoint produced by `01_pretrain_colab.ipynb` and runs the eval split with TTT on and off.

Outputs `submission_*.json` in the ARC Prize Kaggle format so the same predictions can be dry-run against a Kaggle notebook later.

In [ ]:
# Mount Drive (for the checkpoint) and clone the private repo using GITHUB_TOKEN from Colab Secrets.
# One-time setup: Colab sidebar -> Secrets (key icon) -> add GITHUB_TOKEN with `repo` scope.
import os, subprocess, pathlib
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_DIR = pathlib.Path('/content/arc_hybrid_repo')
REPO_URL = 'https://github.com/Nitish05/ARC-AGI.git'

if not REPO_DIR.exists():
    token = userdata.get('GITHUB_TOKEN')
    subprocess.run(
        ['git', 'clone', f'https://{token}@github.com/Nitish05/ARC-AGI.git', str(REPO_DIR)],
        check=True, capture_output=True,
    )
    subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL], check=True)
    del token

%cd /content/arc_hybrid_repo
!pip install -q -r requirements.txt

In [ ]:
# Pull latest from GitHub (run this whenever you've pushed local edits) and
# enable hot-reload so edits to arc_hybrid/*.py take effect on the next cell run
# without needing a kernel restart.
!git -C /content/arc_hybrid_repo pull --ff-only
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from arc_hybrid.utils.config import to_namespace
from arc_hybrid.utils.checkpoint import load_into
from arc_hybrid.model.hybrid import build_hybrid_from_config
from arc_hybrid.data.arc_loader import filter_max_grid, load_split
from arc_hybrid.eval.evaluate import evaluate_split

ckpt_path = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1/slim_final.pt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
cfg = to_namespace(ckpt['config'])
model = build_hybrid_from_config(cfg).to('cuda')
model.load_state_dict(ckpt['model'])
model.eval()
print(f"loaded {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

In [ ]:
from pathlib import Path
tasks = load_split(Path('data/raw/ARC-AGI-2/evaluation'))
tasks = filter_max_grid(tasks, cfg.model.max_grid_size)
print(f'{len(tasks)} eval tasks')

In [ ]:
out_dir = '/content/drive/MyDrive/arc_hybrid_runs/medium_v1/eval'
evaluate_split(model, tasks, use_ttt=False, max_grid=cfg.model.max_grid_size,
               device='cuda', out_dir=out_dir, tag='ttt_off')

In [ ]:
ttt_kwargs = dict(steps=100, lr=5e-4, batch_size=4, n_examples=64,
                  lora_r=8, lora_alpha=16, lora_dropout=0.05)
evaluate_split(model, tasks, use_ttt=True, ttt_kwargs=ttt_kwargs,
               max_grid=cfg.model.max_grid_size, device='cuda',
               out_dir=out_dir, tag='ttt_on')